# Day 9: Momentum — Accelerating Gradient Descent

**Learning Objective**: Implement momentum-based optimization to accelerate training and escape local minima.

Vanilla gradient descent is like a ball stopping and starting at each step. **Momentum** adds inertia — the ball builds up speed in consistent directions and powers through small bumps.

Benefits:
- **Faster convergence** in flat regions
- **Dampens oscillations** in steep regions
- **Escapes shallow local minima**

In [ ]:
import math
import random

## Theory

### Vanilla Gradient Descent
$$\theta_{t+1} = \theta_t - \eta \nabla L(\theta_t)$$

**Problem**: In ravines (narrow valleys), gradients oscillate back and forth.

### Momentum
Add a velocity term that accumulates gradients:

$$v_{t+1} = \beta v_t + \nabla L(\theta_t)$$
$$\theta_{t+1} = \theta_t - \eta v_{t+1}$$

Where:
- $v_t$ = velocity (accumulated momentum)
- $\beta$ = momentum coefficient (typically 0.9)
- $\eta$ = learning rate

### Exponential Moving Average
Each velocity update decays old gradients exponentially:

$$v_t = \beta^0 g_t + \beta^1 g_{t-1} + \beta^2 g_{t-2} + ...$$

Recent gradients matter more; old ones fade.

## Neural Network Code (from Week 1)

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, (self,), f'**{other}')
        def _backward():
            self.grad += (other * self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return Value(other) - self
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other): return Value(other) * self**-1

In [ ]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0)
    
    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()
    
    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
    
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

## Utilities (from Day 8)

In [ ]:
def mse_loss(predictions, targets):
    """Mean Squared Error loss."""
    n = len(predictions)
    return sum((p - t) ** 2 for p, t in zip(predictions, targets)) * (1.0 / n)

def create_batches(X, y, batch_size):
    """Create mini-batches with shuffling."""
    n = len(X)
    indices = list(range(n))
    random.shuffle(indices)
    batches = []
    for i in range(0, n, batch_size):
        batch_idx = indices[i:i+batch_size]
        X_batch = [X[j] for j in batch_idx]
        y_batch = [y[j] for j in batch_idx]
        batches.append((X_batch, y_batch))
    return batches

def train_batched(model, X, y, epochs=100, lr=0.01, batch_size=4, verbose=True):
    """Vanilla SGD training loop (from Day 8)."""
    losses = []
    for epoch in range(epochs):
        batches = create_batches(X, y, batch_size)
        epoch_loss = 0.0
        for X_batch, y_batch in batches:
            preds = [model(x) for x in X_batch]
            loss = mse_loss(preds, y_batch)
            epoch_loss += loss.data
            for p in model.parameters():
                p.grad = 0.0
            loss.backward()
            for p in model.parameters():
                p.data -= lr * p.grad
        losses.append(epoch_loss / len(batches))
    return losses

## SGD with Momentum Optimizer

The key idea: maintain a **velocity** for each parameter that accumulates gradient history.

In [ ]:
class SGDMomentum:
    """SGD with Momentum optimizer."""
    
    def __init__(self, parameters, lr=0.01, momentum=0.9):
        self.parameters = parameters
        self.lr = lr
        self.momentum = momentum
        # Initialize velocities to zero
        self.velocities = [0.0 for _ in parameters]
    
    def step(self):
        """Update parameters using momentum."""
        for i, p in enumerate(self.parameters):
            # Update velocity: v = β*v + gradient
            self.velocities[i] = self.momentum * self.velocities[i] + p.grad
            # Update parameter: θ = θ - lr * v
            p.data -= self.lr * self.velocities[i]
    
    def zero_grad(self):
        """Zero all gradients."""
        for p in self.parameters:
            p.grad = 0.0

## Test 1: Verify Momentum Math

In [ ]:
# Simple test with a single parameter
p = Value(1.0)
p.grad = 0.5

optimizer = SGDMomentum([p], lr=0.1, momentum=0.9)

# Step 1: v = 0.9*0 + 0.5 = 0.5, θ = 1.0 - 0.1*0.5 = 0.95
optimizer.step()
print(f"Step 1: data={p.data:.4f} (expected 0.9500), velocity={optimizer.velocities[0]:.4f}")
assert abs(p.data - 0.95) < 1e-6

# Step 2 (same gradient): v = 0.9*0.5 + 0.5 = 0.95, θ = 0.95 - 0.1*0.95 = 0.855
p.grad = 0.5
optimizer.step()
print(f"Step 2: data={p.data:.4f} (expected 0.8550), velocity={optimizer.velocities[0]:.4f}")
assert abs(p.data - 0.855) < 1e-6

# Step 3: v = 0.9*0.95 + 0.5 = 1.355, θ = 0.855 - 0.1*1.355 = 0.7195
p.grad = 0.5
optimizer.step()
print(f"Step 3: data={p.data:.4f} (expected 0.7195), velocity={optimizer.velocities[0]:.4f}")
assert abs(p.data - 0.7195) < 1e-6

print("\n✅ Momentum math verified! Notice velocity builds up over time.")

## Generic Training Loop with Optimizer

In [ ]:
def train_with_optimizer(model, X, y, optimizer, epochs=100, batch_size=4):
    """Generic training loop with any optimizer."""
    losses = []
    
    for epoch in range(epochs):
        batches = create_batches(X, y, batch_size)
        epoch_loss = 0.0
        
        for X_batch, y_batch in batches:
            # Forward
            preds = [model(x) for x in X_batch]
            loss = mse_loss(preds, y_batch)
            epoch_loss += loss.data
            
            # Backward
            optimizer.zero_grad()
            loss.backward()
            
            # Update
            optimizer.step()
        
        losses.append(epoch_loss / len(batches))
    
    return losses

## Test 2: Momentum Training Reduces Loss

In [ ]:
random.seed(42)

# Dataset: y = 2*x1 - x2
X = [[random.uniform(-1, 1), random.uniform(-1, 1)] for _ in range(50)]
y = [x[0] * 2 - x[1] for x in X]

# Train with momentum
model = MLP(2, [8, 1])
optimizer = SGDMomentum(model.parameters(), lr=0.02, momentum=0.9)

losses = train_with_optimizer(model, X, y, optimizer, epochs=100, batch_size=8)

print(f"Loss: {losses[0]:.4f} → {losses[-1]:.4f}")
assert losses[-1] < losses[0], "Loss should decrease!"
print("✅ Momentum training works!")

## Experiment: Vanilla SGD vs Momentum

Compare convergence speed on a dataset with different scales.

In [ ]:
# Dataset with a "ravine" — different scales in each dimension
random.seed(123)
X = [[random.uniform(-1, 1), random.uniform(-1, 1)] for _ in range(100)]
y = [10*x[0] + 0.1*x[1] for x in X]  # x[0] matters 100x more!

# Vanilla SGD
random.seed(42)
model_sgd = MLP(2, [8, 1])
losses_sgd = train_batched(model_sgd, X, y, epochs=100, lr=0.01,
                           batch_size=16, verbose=False)

# SGD + Momentum
random.seed(42)
model_mom = MLP(2, [8, 1])
opt_mom = SGDMomentum(model_mom.parameters(), lr=0.01, momentum=0.9)
losses_mom = train_with_optimizer(model_mom, X, y, opt_mom,
                                  epochs=100, batch_size=16)

print("Vanilla SGD vs Momentum (every 20 epochs):")
print(f"{'Epoch':>6} | {'Vanilla SGD':>14} | {'Momentum (β=0.9)':>16}")
print("-" * 44)
for i in range(0, 100, 20):
    print(f"{i:6d} | {losses_sgd[i]:14.6f} | {losses_mom[i]:16.6f}")
print(f"{'Final':>6} | {losses_sgd[-1]:14.6f} | {losses_mom[-1]:16.6f}")

print(f"\n📊 Momentum speedup: {losses_sgd[-1]/max(losses_mom[-1], 1e-10):.1f}x lower loss")

## Experiment: Tuning the Momentum Coefficient β

In [ ]:
betas = [0.0, 0.5, 0.9, 0.99]

print("Effect of Momentum Coefficient β:")
print("=" * 50)

for beta in betas:
    random.seed(42)
    model = MLP(2, [8, 1])
    optimizer = SGDMomentum(model.parameters(), lr=0.01, momentum=beta)
    losses = train_with_optimizer(model, X, y, optimizer,
                                  epochs=100, batch_size=16)
    
    status = "✅" if losses[-1] < 1.0 else "⚠️"
    print(f"  β={beta:.2f}: final_loss={losses[-1]:.6f} {status}")

print("\nObservations:")
print("  - β=0.0: Same as vanilla SGD (no momentum)")
print("  - β=0.5: Mild acceleration")
print("  - β=0.9: Standard choice — good balance")
print("  - β=0.99: Very aggressive — can overshoot")

## Summary

Today we implemented:

1. **`SGDMomentum` optimizer** with:
   - Velocity accumulation: `v = β*v + grad`
   - Parameter update: `θ -= lr * v`
   - `zero_grad()` for gradient clearing

2. **Generic training loop** that works with any optimizer

**Key takeaways:**
- Momentum accelerates convergence, especially in ravines
- β=0.9 is the standard default
- Every modern optimizer (Adam, RMSprop) builds on momentum

---

*Previous: [Day 8 — Batching](./day_08_batching.ipynb)*  
*Next: Day 10 — Loss Functions*